# 01: Fast Gradient Sign Method (FGSM) & The Linearity Hypothesis
**Paper:** *Explaining and Harnessing Adversarial Examples* (Goodfellow, Shlens & Szegedy, ICLR 2015)

---

## 1. Mathematical Foundation

### 1.1 The Linearity Hypothesis
Early intuitions hypothesized that adversarial examples were caused by extreme non-linearities and overfitting in deep networks. Goodfellow et al. demonstrated the opposite: **adversarial vulnerability is primarily caused by linear behavior in high-dimensional spaces**.

Consider an input $\mathbf{x} \in \mathbb{R}^n$ with a linear dot-product activation:
$$\mathbf{w}^	op 	ilde{\mathbf{x}} = \mathbf{w}^	op (\mathbf{x} + oldsymbol{\eta}) = \mathbf{w}^	op \mathbf{x} + \mathbf{w}^	op oldsymbol{\eta}$$

If each element of the perturbation $oldsymbol{\eta}$ is constrained by $\|oldsymbol{\eta}\|_\infty \le \epsilon$, the maximum activation increase is achieved by aligning $oldsymbol{\eta}$ with the sign of the weights:
$$oldsymbol{\eta} = \epsilon \mathrm{sign}(\mathbf{w})$$
$$\mathbf{w}^	op oldsymbol{\eta} = \epsilon \sum_{i=1}^n |w_i| = \epsilon \|\mathbf{w}\|_1$$

If $\mathbf{w}$ has $n$ dimensions with an average weight magnitude $m = rac{1}{n} \|\mathbf{w}\|_1$, the activation grows as:
$$\Delta = \epsilon m n$$
In high dimensions (e.g., $n = 784$ for MNIST, or $n = 150,528$ for ImageNet), even an imperceptibly small $\epsilon$ produces a massive change in network activations!

---

### 1.2 The Fast Gradient Sign Method (FGSM)
To maximize the loss $\mathcal{L}(oldsymbol{	heta}, \mathbf{x}, y)$ subject to $\|oldsymbol{\eta}\|_\infty \le \epsilon$, we linearize the loss function around $\mathbf{x}$:
$$\mathcal{L}(oldsymbol{	heta}, \mathbf{x} + oldsymbol{\eta}, y) pprox \mathcal{L}(oldsymbol{	heta}, \mathbf{x}, y) + oldsymbol{\eta}^	op 
abla_{\mathbf{x}} \mathcal{L}(oldsymbol{	heta}, \mathbf{x}, y)$$

The optimal 1-step $L_\infty$ perturbation is:
$$\mathbf{x}_{	ext{adv}} = \mathrm{clamp}\left( \mathbf{x} + \epsilon \cdot \mathrm{sign}\left(
abla_{\mathbf{x}} \mathcal{L}(oldsymbol{	heta}, \mathbf{x}, y)ight), 0, 1 ight)$$


In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Ensure adv_studio library is on path
sys.path.insert(0, os.path.abspath(".."))

from adv_studio.models import SimpleCNN, get_model
from adv_studio.data import get_mnist_loaders, get_sample_digits
from adv_studio.attacks import FGSMAttack, TargetedFGSMAttack
from adv_studio.evaluation import compute_robust_accuracy, compute_distortion_metrics
from adv_studio.visualization import plot_adversarial_gallery, plot_robustness_curves

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Execution device: {device}")


## 2. Load Pretrained Clean Model & Dataset


In [ ]:
train_loader, test_loader = get_mnist_loaders(data_dir="../DATA", batch_size=64)

# Load clean-trained model checkpoint
model = get_model("simple_cnn", pretrained_path="../checkpoints/mnist_cnn.pth", device=device)
model.eval()

# Quick evaluation of clean accuracy
clean_metrics = compute_robust_accuracy(model, test_loader, device=device, max_batches=15)
print(f"[+] Clean Test Accuracy: {clean_metrics['clean_accuracy']*100:.2f}%")


## 3. FGSM Attack Across Perturbation Budgets ($\epsilon$)


In [ ]:
epsilons = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4]
fgsm_accuracies = []

for eps in epsilons:
    if eps == 0.0:
        fgsm_accuracies.append(clean_metrics['clean_accuracy'])
    else:
        attack = FGSMAttack(model, epsilon=eps, device=device)
        res = compute_robust_accuracy(model, test_loader, attack=attack, device=device, max_batches=15)
        fgsm_accuracies.append(res['robust_accuracy'])
        print(f"Epsilon: {eps:.2f} | Robust Acc: {res['robust_accuracy']*100:.2f}% | ASR: {res['attack_success_rate']*100:.2f}%")


## 4. Visualizing Robustness Degradation Curve


In [ ]:
fig = plot_robustness_curves(
    epsilons=epsilons,
    results_by_model={"Clean Model (SimpleCNN)": fgsm_accuracies},
    title="FGSM: Epsilon vs Adversarial Robustness on MNIST"
)
plt.show()


## 5. Adversarial Gallery: Clean vs Perturbation vs Adversary


In [ ]:
sample_imgs, sample_lbls = get_sample_digits(data_dir="../DATA", count=5)
sample_imgs, sample_lbls = sample_imgs.to(device), sample_lbls.to(device)

attack = FGSMAttack(model, epsilon=0.25, device=device)
adv_imgs = attack.generate(sample_imgs, sample_lbls)

with torch.no_grad():
    clean_logits = model(sample_imgs)
    adv_logits = model(adv_imgs)
    clean_preds = clean_logits.argmax(dim=1)
    adv_preds = adv_logits.argmax(dim=1)
    clean_probs = torch.softmax(clean_logits, dim=1)
    adv_probs = torch.softmax(adv_logits, dim=1)

fig = plot_adversarial_gallery(
    sample_imgs, adv_imgs, sample_lbls,
    clean_preds, adv_preds, clean_probs, adv_probs,
    num_samples=5
)
plt.show()
